### POC ML for Futures Trading


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
# Yahoo Finance – continuous front-month (EOD and recent intraday)
es_eod = yf.download("ES=F", start="2015-01-01")     # daily OHLCV
es_intraday = yf.download("ES=F", period="7d", interval="1m")  # minute (limited window)
print(es_eod.tail(), es_intraday.tail())

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Price         Close     High      Low     Open   Volume
Ticker         ES=F     ES=F     ES=F     ES=F     ES=F
Date                                                   
2026-01-16  6976.75  7007.00  6960.50  6986.75  1297479
2026-01-20  6829.50  6935.00  6822.25  6918.25  2391547
2026-01-21  6910.00  6945.25  6814.50  6839.00  2200553
2026-01-22  6945.00  6969.00  6911.25  6920.00  1336659
2026-01-23  6945.75  6964.00  6924.75  6938.00  1336659 Price                        Close     High      Low     Open Volume
Ticker                        ES=F     ES=F     ES=F     ES=F   ES=F
Datetime                                                            
2026-01-23 21:55:00+00:00  6934.25  6934.75  6933.75  6934.25    184
2026-01-23 21:56:00+00:00  6933.75  6934.75  6933.75  6934.25    137
2026-01-23 21:57:00+00:00  6934.00  6934.25  6933.50  6934.00    298
2026-01-23 21:58:00+00:00  6933.75  6934.25  6933.50  6934.25    217
2026-01-23 21:59:00+00:00  6933.75  6934.25  6933.50  6933.75      0


In [7]:
# Download futures data from yfinance
# Download E-Mini S&P 500 futures (ES) - daily data
es_futures = yf.download("ES=F", start="2020-01-01", end="2026-01-24")


[*********************100%***********************]  1 of 1 completed


In [14]:
es_futures

Price,Close,High,Low,Open,Volume
Ticker,ES=F,ES=F,ES=F,ES=F,ES=F
Date,,,,,
2020-01-02,3259.00,3261.75,3234.25,3237.00,1416241
2020-01-03,3235.50,3263.50,3206.75,3261.00,1755057
2020-01-06,3243.50,3249.50,3208.75,3220.25,1502748
2020-01-07,3235.25,3254.50,3226.00,3243.50,1293494
2020-01-08,3260.25,3267.75,3181.00,3231.75,2279138
...,...,...,...,...,...
2026-01-16,6976.75,7007.00,6960.50,6986.75,1297479
2026-01-20,6829.50,6935.00,6822.25,6918.25,2391547


In [17]:
# Convert to single index format
# Remove MultiIndex if present and flatten columns
if isinstance(es_futures.columns, pd.MultiIndex):
    es_futures.columns = es_futures.columns.get_level_values(0)

# Ensure the index is a proper DatetimeIndex
es_futures.index = pd.to_datetime(es_futures.index)
es_futures.index.name = 'Date'

# Remove any NaN rows
es_futures = es_futures.dropna()

# Standardize column names (keep existing number of columns)
es_futures.columns = [col.replace(' ', '_') for col in es_futures.columns]

print("Cleaned es_futures DataFrame:")
print(f"Shape: {es_futures.shape}")
print(f"Index type: {type(es_futures.index)}")
print(f"Columns: {list(es_futures.columns)}")
print(f"\nFirst few rows:")
print(es_futures.head())
print(f"\nLast few rows:")
print(es_futures.tail())
print(f"\nData info:")
print(es_futures.info())

Cleaned es_futures DataFrame:
Shape: (1526, 5)
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Columns: ['Close', 'High', 'Low', 'Open', 'Volume']

First few rows:
              Close     High      Low     Open   Volume
Date                                                   
2020-01-02  3259.00  3261.75  3234.25  3237.00  1416241
2020-01-03  3235.50  3263.50  3206.75  3261.00  1755057
2020-01-06  3243.50  3249.50  3208.75  3220.25  1502748
2020-01-07  3235.25  3254.50  3226.00  3243.50  1293494
2020-01-08  3260.25  3267.75  3181.00  3231.75  2279138

Last few rows:
              Close     High      Low     Open   Volume
Date                                                   
2026-01-16  6976.75  7007.00  6960.50  6986.75  1297479
2026-01-20  6829.50  6935.00  6822.25  6918.25  2391547
2026-01-21  6910.00  6945.25  6814.50  6839.00  2200553
2026-01-22  6945.00  6969.00  6911.25  6920.00  1336659
2026-01-23  6945.75  6964.00  6924.75  6938.00  1336659

Data info:
<class

In [20]:
df = es_futures.reset_index()
df

,Date,Close,High,Low,Open,Volume
0,2020-01-02,3259.00,3261.75,3234.25,3237.00,1416241
1,2020-01-03,3235.50,3263.50,3206.75,3261.00,1755057
2,2020-01-06,3243.50,3249.50,3208.75,3220.25,1502748
3,2020-01-07,3235.25,3254.50,3226.00,3243.50,1293494
4,2020-01-08,3260.25,3267.75,3181.00,3231.75,2279138
...,...,...,...,...,...,...
1521,2026-01-16,6976.75,7007.00,6960.50,6986.75,1297479
1522,2026-01-20,6829.50,6935.00,6822.25,6918.25,2391547
1523,2026-01-21,6910.00,6945.25,6814.50,6839.00,2200553
1524,2026-01-22,6945.00,6969.00,6911.25,6920.00,1336659


In [21]:
# Calculate standard deviation across Open, Close, High, Low for each row
df['OHLC_Std'] = df[['Open', 'High', 'Low', 'Close']].std(axis=1)

# Display the result
print("DataFrame with OHLC Standard Deviation:")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows with new column:")
print(df[['Open', 'High', 'Low', 'Close', 'OHLC_Std']].head(10))
print(f"\nStatistics for OHLC_Std:")
print(df['OHLC_Std'].describe())

DataFrame with OHLC Standard Deviation:
Shape: (1526, 7)

First few rows with new column:
      Open     High      Low    Close   OHLC_Std
0  3237.00  3261.75  3234.25  3259.00  14.377355
1  3261.00  3263.50  3206.75  3235.50  26.505797
2  3220.25  3249.50  3208.75  3243.50  19.219131
3  3243.50  3254.50  3226.00  3235.25  12.123282
4  3231.75  3267.75  3181.00  3260.25  39.313046
5  3261.25  3276.75  3257.75  3276.00   9.851766
6  3275.50  3287.00  3260.75  3264.75  11.780988
7  3265.75  3291.00  3265.50  3289.75  14.298893
8  3289.25  3296.75  3275.25  3288.00   8.921825
9  3287.75  3299.00  3277.75  3293.75   9.118148

Statistics for OHLC_Std:
count    1526.000000
mean       33.267906
std        22.668005
min         4.942903
25%        18.855570
50%        27.686394
75%        41.279658
max       331.857336
Name: OHLC_Std, dtype: float64
